# 模組 6：用 LLM 把財經新聞打成情緒分數

把一則財經新聞讀進來 → 讓 LLM 判斷利多／利空／中性、給一個分數 → 批次算出「每檔股票每天一個情緒分數」，存成一欄餵回選股模型。這一欄就是下一個 Lab（模組 9 合流）開頭要載入的那欄。

> ## ⚠️ 金融免責（先讀）
> 今天做的「財經新聞情緒打分機」純教學示範，新聞、情緒分數**全部虛構**，不是投資建議。單一情緒訊號在真實股市的解釋力很低。今天學的是「怎麼用 LLM 把新聞文字算成一個能存進表格的分數、怎麼接回模型」，不是「看到利多就買股票」。機敏財務資料用 Ollama 本地不外傳、最終靠人判斷（human-in-the-loop）。

## 今天這條線

你已經做完數據路（KNN／決策樹／GA）：用**財務數字**猜漲跌。但有些影響漲跌的東西，財務欄位裡根本沒有——比如「今天新聞說這家公司是利多還是利空」。今天進入**文字路**：用 LLM 把新聞讀成一個情緒分數。

- **A** 為什麼把新聞變分數 → zero-shot 直接問（看 LLM 回的自由文字、很亂）
- **B** few-shot ＋ 要求回結構化格式 → 拿到能存表格的 `{label, score}` → 包成 `score_news()`
- **C** 批次跑一批新聞 → 每檔每天平均 → 一欄 `news_sent`（存 CSV，模組 9 讀）
- **D** 收尾：文字路第一塊完成、接模組 9

In [ ]:
# 先跑這一格：一次裝齊今天要用的套件（已裝好的會直接跳過）
!pip install -q langchain-ollama pandas

## 🟦 A 段：為什麼把新聞變分數 ＋ zero-shot 直接問

### 環境就緒：接上 Ollama Cloud（沿用課程 1 那支設定）

先貼 key、確認雲端接得上。這格跟課程 1 一字不差，不重教安裝。

In [ ]:
import os

# 👇 把引號中間換成老師給你的 OLLAMA key（整段貼進去，前後別留空白）
os.environ["OLLAMA_API_KEY"] = "在這裡貼上你的 OLLAMA key"

key = os.environ.get("OLLAMA_API_KEY")
if key and key != "在這裡貼上你的 OLLAMA key":
    print("✅ OLLAMA_API_KEY 已設定（長度", len(key), "個字元）")
else:
    print("❌ 還沒貼 key → 把上面那行引號中間換成你的 key，再重跑這一格")
# 🔒 只印長度、不印 key 本身——貼了 key 的 notebook 別上傳 GitHub、別傳給別人。

In [ ]:
from langchain_ollama import ChatOllama

# TODO：填建立雲端 LLM 的類別（課程 1 用的那個，字首大寫、後面接一對括號）
llm = ____(
    model="gemma4:cloud",           # 雲端模型要帶 :cloud tag；漏了會去找本機同名模型 → model not found
    base_url="https://ollama.com",  # 打到雲端（不寫這行就是打你自己電腦的 localhost）
    client_kwargs={"headers": {"Authorization": f"Bearer {os.environ['OLLAMA_API_KEY']}"}},
)
# TODO：填「把問題送出去、拿回生成」的方法（課程 1 用過），後面 .content 才拿得到文字
print(llm.____("用一句繁體中文說你準備好了").content)   # 通了 → 雲端可達
# 💡 若 401：key 沒設或貼錯；若 model not found：漏了 :cloud

### 為什麼用 LLM，不用「關鍵字字典」數詞？

最土的做法是列「利多詞表／利空詞表」數哪邊多。但它看不懂**否定**（「**沒有**虧損」被當利空）、看不懂**語境**（「大跌，逢低買進好時機」）。LLM 讀懂整句意思再判，還會中文、zero-shot 不用訓練。

| | 關鍵字字典 | LLM |
|---|---|---|
| 怎麼判 | 數利多／利空詞哪邊多 | 讀懂整句意思 |
| 「沒有虧損」 | ❌ 判利空（錯） | ✅ 懂是偏正面 |
| 要不要訓練 | 手動維護詞表 | 不用、會中文 |

### 準備幾則「虛構財經新聞」

刻意編成明顯利多／利空／中性各一則（教學用乾淨資料，等等看得出 LLM 判得對不對）。🚨 全部虛構，不是任何真實公司或新聞。

In [ ]:
# 【虛構・教學用】財經新聞（非真實個股、非真實新聞）
news_good = "（虛構示意）某科技公司宣布拿下海外大廠年度大單，預估營收將顯著成長，市場看好後市。"
news_bad = "（虛構示意）某公司遭爆財務報表造假、高層請辭，主管機關已介入調查，投資人信心受創。"
news_neutral = "（虛構示意）某集團今日召開例行股東會，董事會組成維持不變，營運大致符合先前展望。"
for n in [news_good, news_bad, news_neutral]:
    print(n)

### zero-shot：直接問，不給格式

**zero-shot（零範例提示）** ＝直接問 LLM、不給任何範例（課程 1 學過）。先用最省事的方式問一則，看它回什麼。

In [ ]:
# zero-shot：直接問，不給任何範例、不規定格式
prompt = f"""
這則財經新聞對該公司是利多還是利空？

新聞：

{____}

"""
# TODO：大括號裡填要判斷的那則新聞變數（明顯利多那則）
print(llm.invoke(prompt).content)

> **🔑 判得對，但回的是「一大段自由文字」**——這次「利多」開頭、下次可能「我認為這是正面消息」、再下次條列。你沒辦法穩定從這堆文字抽出一個乾淨分數，而且它**沒給 0~1 的數字**。模型要的是一欄數字，不是一段話。B 段解決。

## 🟩 B 段：few-shot ＋ 要求回結構化格式 ＋ 寫出 `score_news()`

兩個工具把「亂回的自由文字」升級成「能存表格的乾淨分數」：**few-shot 給範例穩住格式** ＋ **要求回一種機器能直接解析的格式**。

- **few-shot（少量範例提示）** ＝先給 1~2 個「新聞→輸出」範例，LLM 會模仿格式。
- 我們要的輸出是「鍵: 值」的結構化格式 `{"label":"利多","score":0.8}`，Python 能直接解析成 `dict`。
- 建 `llm` 時加一個參數，強迫 Ollama 回合法的結構化格式。

In [ ]:
# 跟課程 1 同設定，多加一個參數強迫回合法結構化格式
llm_json = ChatOllama(
    model="gemma4:cloud",
    base_url="https://ollama.com",
    format=____,                                  # TODO：填強迫回合法結構化格式的值（那種格式的名字，小寫、加引號）
    client_kwargs={"headers": {"Authorization": f"Bearer {os.environ['OLLAMA_API_KEY']}"}},
)

In [ ]:
# few-shot prompt：給規則 + 2 個範例（示範我要的格式），再丟新聞
PROMPT_TMPL = """你是金融新聞情緒分析助手。判斷新聞對該公司是「利多」「利空」或「中性」，
並給 0~1 的分數（越接近 1＝越正面/利多，越接近 0＝越負面/利空，0.5＝中性）。
只回一個結構化格式：{{"label": "利多/利空/中性", "score": 0~1 的數字}}，不要多餘文字。

範例 1
新聞：公司營收創新高、獲利大幅成長。
輸出：{{"label": "利多", "score": 0.85}}

範例 2
新聞：公司爆發弊案、高層遭調查。
輸出：{{"label": "利空", "score": 0.15}}

現在請判斷這則新聞：
新聞：{news}
輸出："""

# TODO：把模板裡的 {news} 換成明顯利多那則新聞 → 用 .format(佔位名=新聞變數)
raw = llm_json.invoke(PROMPT_TMPL.format(____)).content
print("LLM 原始回傳：", raw)

> **🔑 比 A 段乾淨多了。** 但 LLM 偶爾還是會多包一層 ```json 圍欄、或前面加一句廢話。所以**不直接信 LLM**——下一格用一個穩健的 parser 把分數穩穩抽出來，抽不到就退回一個安全預設。

In [ ]:
import json, re

def parse_sentiment(text):
    """從 LLM 回傳穩健抽出 {label, score}；抓不到就退回硬比對關鍵字。"""
    cleaned = re.sub(r"```(json)?", "", text).strip()      # 去掉 ```json 圍欄
    m = re.search(r"\{.*?\}", cleaned, re.DOTALL)          # 找第一個 {...}
    if m:
        try:
            d = json.loads(m.group())
            score = max(0.0, min(1.0, float(d.get("score", 0.5))))   # 把分數夾在 0~1
            return {"label": d.get("label", "中性"), "score": score}
        except (json.JSONDecodeError, ValueError, TypeError):
            pass
    # 到這裡＝上面沒解析成功 → 改用「硬比對」：直接看字串裡有沒有「利多」「利空」兩個字
    if "利多" in text:
        return {"label": "利多", "score": 0.7}
    if "利空" in text:
        return {"label": "利空", "score": 0.3}
    return {"label": "中性", "score": 0.5}

# 對四種輸入各測一次（這格純 Python，每次跑都一樣）
print(parse_sentiment('{"label": "利多", "score": 0.82}'))             # 乾淨結構化
print(parse_sentiment('```json\n{"label":"利空","score":0.15}\n```'))  # 圍欄包覆
print(parse_sentiment('我覺得是這樣：{"label":"中性","score":0.5}'))    # 前面有廢話
print(parse_sentiment('看不懂的一段話，但有提到利多'))                  # 退回硬比對

> **🔑 這格是純 Python、決定性的**（每次一樣，所以上面 LLM 回傳標「示意」、這裡是固定值）。它把 LLM 的「亂」吸收掉：去圍欄 → 找 `{...}` → 解析、把 score 夾在 0~1。
> **最後那三行是「硬比對」**：如果前面連 `{...}` 都抓不到，就**直接看字串裡有沒有「利多」「利空」兩個字**來給分——這是最後的安全網，永遠回得出東西、不會炸。硬比對很粗（不讀語意、只認字），所以只當「解析失敗」時的退路。`\{.*?\}` ＝「找第一個大括號包住的東西」，不用會寫正則，知道它在撈那段結構化資料即可。

### 包成 `score_news()`：吃一則新聞 → 回乾淨的 `{label, score}`

In [ ]:
def score_news(news):
    """吃一則新聞文字，回 {'label':..., 'score':...}（0~1）。"""
    # TODO：① 用剛剛那個「會回結構化格式」的 llm 物件；② 把模板的 {news} 換成傳進來的 news
    raw = ____.invoke(PROMPT_TMPL.format(____)).content
    return parse_sentiment(raw)

# 對三則虛構新聞各跑一次
for n in [news_good, news_bad, news_neutral]:
    print(score_news(n), "←", n[:24], "...")

> **🔑 你做出了 `score_news()`——今天第一個交付物。** LLM 算的 label/score 每次略不同（示意），但格式被 few-shot＋結構化＋parser 鎖死了：永遠回得出 `{label, score}`、永遠能存進表格。

### 📝 小作業 A：換一則你自己編的虛構新聞，丟給 `score_news()`

自己編一則明顯利多或利空的虛構新聞（🚨 不要用真實公司），看 LLM 判得對不對。

In [ ]:
# 📝 小作業 A：把你自己編的虛構新聞丟進上面做好的那個函式
my_news = "（虛構示意）某公司宣布調高全年財測，並將擴大海外投資，法人紛紛看好。"
# TODO：呼叫上面做好的那個「吃一則新聞 → 回 {label, score}」的函式
print(____(my_news))
# 💡 明顯利多 → label 應大致回「利多」、score 偏高（>0.5）；數值每次略不同是正常的

## 🟧 C 段：批次跑一批新聞 → 每檔每天一個 `news_sent`

**這段是今天的課程重點，也是文字路交給數據路的交接點。** `score_news()` 一次只吃一則，但一檔股票一天可能有好幾則新聞——模型要的是「這檔這天的整體情緒」一個數字。兩步做到：

1. **`to_signed`**：把 `{label, score}` 合成一個有正負的方向分（利多正／利空負／中性 0），範圍 [-1, 1]。
2. **每檔每天平均**：同股同日多則新聞平均成一個 → 這就是 `news_sent`。

In [ ]:
def to_signed(p):                      # 把 {label, score} 合成 [-1,1] 方向分
    if p["label"] == "利多":
        return p["score"]              # 利多 → 保持正（看好）
    if p["label"] == "利空":
        return -p["score"]             # 利空 → 翻成負（看壞）
    return 0.0                         # 中性 → 0

# 純 Python、決定性
print(to_signed({"label": "利多", "score": 0.82}))   # +0.82
print(to_signed({"label": "利空", "score": 0.14}))   # -0.14
print(to_signed({"label": "中性", "score": 0.5}))    #  0.0

> **🔑 利多保持正、利空翻負、中性歸零**——把「0~1 的分數（只有大小）」變成「-1~1 的方向分（有正負）」。這就是 `news_sent` 的定義。

### 準備一批「(股票, 日期, 新聞)」，批次跑 `score_news`

In [ ]:
import pandas as pd

# 【虛構・教學用】一批新聞：每列 =(股票, 第幾天, 一則新聞)。同股同日可有多則。
# 🚨 全部虛構、非真實個股/新聞。股票用代號 A/B，日期用 day 1/2 示意。
raw_news = [
    ("A", 1, "（虛構）A 公司接獲大型訂單，營收看增。"),
    ("A", 1, "（虛構）分析師上調 A 公司目標價。"),
    ("A", 2, "（虛構）A 公司產線傳出小幅延誤，影響有限。"),
    ("B", 1, "（虛構）B 公司遭調查、高層請辭，投資人信心受創。"),
    ("B", 2, "（虛構）B 公司召開例行股東會，營運符合展望。"),
]

records = []
for stock, day, news in raw_news:
    p = score_news(news)                # B 段做的函式：吃新聞 → {label, score}
    signed = to_signed(p)               # 合成方向分
    records.append({"stock": stock, "day": day,
                    "label": p["label"], "score": p["score"], "signed": signed})
df_news = pd.DataFrame(records)
print(df_news)

> **🔑 注意 A 股第 1 天有「兩則」新聞（兩列）。** 模型要「A 股第 1 天的整體情緒」一個數字。`label`/`score` 是 LLM 算的（示意）；`signed` 是 `to_signed` 算的（決定性）。

### 一步一步組出 `news_sent`（一次一個 pandas 動作）

下面把「壓成每檔每天一個分數」拆成三個 pandas 動作，每步都印出來看它變成什麼樣子。

In [ ]:
# 第 1 步：把「同檔同天」分成一組，再把每組的 signed 平均成一個數字
# TODO：① 填「分組」的動作（吃一串欄名）；② 填「平均」的動作
avg = df_news.____(["stock", "day"])["signed"].____()
print(avg)

> **`groupby` 分組 + `mean` 平均**：本來 A 股第 1 天有兩列，現在合成一個數字（兩則平均）。左邊的 `(stock, day)` 現在是**索引**（掛在左側、不是普通欄位）——這叫 Series，還不是好用的表格。

In [ ]:
# 第 2 步：把索引 (stock, day) 放回成正常欄位
# TODO：填「把索引變回普通欄位」的動作（名字有 index）
avg = avg.____()
print(avg)

> `(stock, day)` 本來掛在左邊當索引，這步把它們**放回成正常欄位**，變回一張正常的表（DataFrame）。

In [ ]:
# 第 3 步：把欄名 signed 改成 news_sent（模組 9 要讀的欄名）
# TODO：填「改欄名」的動作（吃 columns={舊名: 新名}）
daily = avg.____(columns={"signed": "news_sent"})
print(daily)

> **🔑 這張 `daily` 的 `news_sent` 欄就是模組 9 開頭載入的那欄！** 每檔股票（`stock`）每天（`day`）一個情緒分數（float、範圍 [-1,1]）。A 股第 1 天兩則利多平均成正值、B 股第 1 天利空為負。今天你親手把「新聞文字」變成了「模型能吃的一欄數字」。
> ⚠️ 這欄要用「該日（含）以前」的新聞算，**不能混進未來新聞**——模組 9 會嚴防這種偷看未來（leakage）。今天先把分數綁對 `(stock, day)`。

### 存成 CSV（模組 9 直接讀這個檔）

In [ ]:
# TODO：填「把 DataFrame 存成 CSV 檔」的動作（名字有 csv）
daily.____("news_sent.csv", index=False)
print("已存 news_sent.csv（模組 9 開頭會載入這欄）")
print("欄位：", list(daily.columns))

### 📝 小作業 B：加一檔 C 股的新聞，重算 `news_sent`

在 `raw_news` 再加兩三則 C 股的虛構新聞（同一天可多則），重跑批次 → 三步聚合，看 C 股那天的 `news_sent`。

In [ ]:
# 📝 小作業 B：加兩則 C 股的虛構新聞，重算 news_sent
raw_news_2 = raw_news + [
    ("C", 1, "（虛構）C 公司新產品熱賣，訂單湧入。"),
    ("C", 1, "（虛構）C 公司獲政府補助，擴廠計畫啟動。"),
]
records2 = []
for stock, day, news in raw_news_2:
    p = score_news(news)
    records2.append({"stock": stock, "day": day, "signed": to_signed(p)})
# TODO：照上面三步（分組平均 → 放回欄位 → 改欄名）把 records2 壓成每檔每天一個 news_sent
avg2 = pd.DataFrame(records2).____(["stock", "day"])["signed"].____().____()
daily2 = avg2.____(columns={"signed": "news_sent"})
print(daily2)
# 💡 C 股兩則都是利多 → C/1 的 news_sent 應該是明顯的正值

### 業界也有「專門訓練好的情緒模型」：FinBERT（只講、今天不動手）

**FinBERT**（`ProsusAI/finbert`）是 BERT 在金融語料上微調好的情緒模型，輸出 positive/negative/neutral 三類。它跟我們的 LLM-prompt 是兩條路線：

| | FinBERT | LLM-prompt（今天做法） |
|---|---|---|
| 本質 | 專門訓練的監督模型 | 通用大模型＋Prompt，zero/few-shot 不用訓練 |
| 語言 | 英文為主 | 會中文 |
| 輸出 | 固定 3 標籤 | 彈性：自訂 label＋0~1 分數 |
| 在哪跑 | 下載模型、地端 | Ollama（本地/雲端），機敏資料不外傳 |

做**繁中**財經新聞＋要**彈性分數** → 今天選 LLM-prompt。FinBERT 列出來讓你知道還有這條路，今天不要求裝。

## 🟫 D 段：收尾 — 文字路第一塊完成、接模組 9

今天你做了一台「把財經新聞變成情緒分數」的機器，而且把它接到了模型要的形狀：

```mermaid
flowchart LR
    NEWS[虛構財經新聞<br/>文字] --> ZERO[zero-shot 直接問<br/>自由文字·亂]
    ZERO --> FEW[few-shot + 結構化<br/>label+score]
    FEW --> FUNC[score_news 函式<br/>+ 穩健 parser]
    FUNC --> BATCH[批次跑一批新聞<br/>to_signed → 分組平均]
    BATCH --> SENT[news_sent 欄<br/>每股每日一個分數<br/>存 CSV]
    SENT --> M9[模組9 合流<br/>當第4特徵餵回決策樹]
```

**文字路三招：① 情緒打分（今天）→ ② 財報抽取（下個 Lab）→ ③ 財報 RAG。** 一直在做同一件事：把「文字」變成「模型／人能用的結構化東西」。

> **接下一個 Lab（模組 9 合流）：** 今天做的 `news_sent` 欄，模組 9 開頭第一件事就是載入它、當「第 4 個特徵」塞進你那棵決策樹，親眼看「加情緒前 vs 加情緒後」準不準。那就是收束句：「**LLM ＝ 文字特徵工廠**，把以前抓不到的文字變成模型能吃的一欄數字。」模組 9 還會嚴守 leakage 鐵則（用 `.shift(1)` 只取前一日新聞）。

> ## 🚫 收尾免責
> 今天的情緒打分機純教學、全虛構，不是投資建議。情緒分析在金融上是**輔助參考**，單一訊號解釋力很低。金融三鐵則：① 數字回查原文（LLM 會幻覺）；② 不可當投資建議（最終決策靠人）；③ 機敏資料不外傳（用 Ollama 本地）。

> 🎓 iPAS / AI-901 考點：情緒分析、NLP、Prompt 工程（zero-shot／few-shot）、結構化輸出、特徵工程。

## 🛟 附錄：最小可跑核心

某格卡住時，這幾行（純 Python、不需 Ollama）能跑，就代表整個 lab 的「決定性骨幹」沒問題。

In [ ]:
import json, re
import pandas as pd

def parse_sentiment(text):
    cleaned = re.sub(r"```(json)?", "", text).strip()
    m = re.search(r"\{.*?\}", cleaned, re.DOTALL)
    if m:
        try:
            d = json.loads(m.group())
            s = max(0.0, min(1.0, float(d.get("score", 0.5))))
            return {"label": d.get("label", "中性"), "score": s}
        except (json.JSONDecodeError, ValueError, TypeError):
            pass
    if "利多" in text:
        return {"label": "利多", "score": 0.7}
    if "利空" in text:
        return {"label": "利空", "score": 0.3}
    return {"label": "中性", "score": 0.5}

def to_signed(p):
    if p["label"] == "利多":
        return p["score"]
    if p["label"] == "利空":
        return -p["score"]
    return 0.0

fake = [("A", 1, '{"label":"利多","score":0.8}'),
        ("A", 1, '{"label":"利多","score":0.7}'),
        ("B", 1, '{"label":"利空","score":0.2}')]
rows = []
for s, d, t in fake:
    rows.append({"stock": s, "day": d, "signed": to_signed(parse_sentiment(t))})
out = (pd.DataFrame(rows).groupby(["stock", "day"])["signed"]
                         .mean().reset_index().rename(columns={"signed": "news_sent"}))
print(out)   # 應印 A/1 → 0.75、B/1 → -0.2